# Canadian Cheese Directory + Provincial Weather Analysis
**Data Analyst Intern Assessment – Canadian Sheep Federation**

**Author:** Joshua van Zyll de Jong

**Date:** May 12th,2026

This notebook analyzes the Canadian Cheese Directory cross-referenced with historical provincial temperature data. It demonstrates data loading, cleaning, aggregation, merging, visualization, and inference.

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported")
print("Files in folder:", os.listdir('.'))

✅ Libraries imported
Files in folder: ['.DS_Store', 'spy_daily.csv', 'PyCharmMiscProject.iml', '.venv', 'outputs', '.idea']


In [7]:
cheese = pd.read_csv('cheese_data.csv')
temp = pd.read_csv('Canada_Temperature_Data.csv', low_memory=False)

print("✅ Files loaded successfully!")
print("Cheese rows:", cheese.shape[0])
print("Temp rows:", temp.shape[0])

FileNotFoundError: [Errno 2] No such file or directory: 'cheese_data.csv'

In [ ]:
cheese = cheese.dropna(subset=['ManufacturerProvCode']).copy()
cheese['MoisturePercent'] = pd.to_numeric(cheese['MoisturePercent'], errors='coerce')

temp['Tm'] = pd.to_numeric(temp['Tm'], errors='coerce')
temp = temp.dropna(subset=['Tm'])

print("✅ Data cleaned")

In [ ]:
cheese_summary = cheese.groupby('ManufacturerProvCode').agg(
    num_cheeses=('CheeseId', 'count'),
    avg_moisture=('MoisturePercent', 'mean'),
    most_common_milk=('MilkTypeEn', lambda x: x.mode().iloc[0] if not x.empty else 'Unknown')
).reset_index()

print(cheese_summary.sort_values('num_cheeses', ascending=False))

In [ ]:
avg_temp = temp.groupby('Prov')['Tm'].mean().reset_index()
avg_temp = avg_temp.rename(columns={'Tm': 'avg_mean_temp_c'})
print(avg_temp.sort_values('avg_mean_temp_c').head(10))

In [ ]:
merged = pd.merge(
    cheese_summary,
    avg_temp,
    left_on='ManufacturerProvCode',
    right_on='Prov',
    how='left'
).drop(columns=['Prov'])
print("✅ Merge done. Shape:", merged.shape)

In [ ]:
plt.figure(figsize=(14, 7))
sns.barplot(
    data=cheese_summary.sort_values('num_cheeses', ascending=False),
    x='ManufacturerProvCode',
    y='num_cheeses',
    hue='most_common_milk',
    palette='Set2'
)
plt.title('Number of Cheese Varieties by Province', fontsize=16)
plt.xlabel('Province')
plt.ylabel('Number of Cheeses')
plt.xticks(rotation=45)
plt.legend(title='Most Common Milk Type')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=merged,
    x='avg_mean_temp_c',
    y='num_cheeses',
    hue='ManufacturerProvCode',
    size='avg_moisture',
    sizes=(50, 400),
    palette='tab10'
)
plt.title('Average Temperature vs Number of Cheese Varieties', fontsize=16)
plt.xlabel('Average Mean Temperature (°C)')
plt.ylabel('Number of Cheese Varieties')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()